In [1]:
pip install pandas numpy scipy openpyxl xlrd matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\HP'\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# 1) Imports
import os, re, glob, warnings, unicodedata
from pathlib import Path
from typing import Dict, Optional, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy import trapezoid
from scipy import stats

warnings.filterwarnings("ignore", message=".*CODEPAGE.*")   # silence old .xls codepage chatter
plt.rcParams["figure.dpi"] = 120

# %matplotlib inline


In [3]:
FOLDER_STRESS       = r"./stress"               # folder of stress sessions
FOLDER_CONVENTIONAL = r"./medit_conventional"   # folder of conventional meditation
FOLDER_CUSTOM       = r"./VR meditation"        # folder of VR/custom meditation


In [11]:
# -----------------------------
# Optional: candidate mapping (file→subject_id)
# -----------------------------
# If you want to use the Excel list to force subject IDs, set the path here.
CANDIDATE_LIST = r"./Candidate List.xlsx"   # "" to disable mapping entirely

FILEKEY_TO_SUBJECT: Dict[str, str] = {}

def _find_like(cols, wanted):
    """Return the first column whose lowercase equals or contains 'wanted'."""
    wl = wanted.lower()
    for c in cols:
        s = str(c).strip()
        if s.lower() == wl:
            return s
    for c in cols:
        s = str(c).strip()
        if wl in s.lower():
            return s
    return None

if CANDIDATE_LIST:
    # Try several header rows because your file has a title line above the headers
    mapping_loaded = False
    last_err = None
    for hdr in (0, 1, 2, 3, 4):
        try:
            md = pd.read_excel(CANDIDATE_LIST, header=hdr)
            cols = [str(c).strip() for c in md.columns]
            col_code = _find_like(cols, "Candidate code") or _find_like(cols, "code")
            if not col_code:
                continue
            # Build subject_id from the code column (normalize to alnum)
            md = md[[col_code]].dropna()
            md["__filekey__"] = md[col_code].astype(str).map(make_filekey)
            md["__subject__"] = md[col_code].astype(str).map(_norm)
            FILEKEY_TO_SUBJECT = dict(zip(md["__filekey__"], md["__subject__"]))
            print(f"[Mapping] Loaded {len(FILEKEY_TO_SUBJECT)} IDs from header row {hdr} "
                  f"(code column = '{col_code}').")
            mapping_loaded = True
            break
        except Exception as e:
            last_err = e
            continue
    if not mapping_loaded:
        print("Could not locate a 'Candidate code' column in the mapping file.")
        if last_err:
            print("Last error:", type(last_err).__name__, last_err)

# Use mapping if present; otherwise use filename stem
def extract_subject_id(path: str) -> str:
    key = make_filekey(Path(path).stem)
    sid = FILEKEY_TO_SUBJECT.get(key)
    if sid:
        return sid[:24]
    return key[:24]


[Mapping] Loaded 13 IDs from header row 1 (code column = 'Candidate code').


In [12]:
# Epoch length in seconds (0 for whole-session features; recommended 60)
EPOCH_S = 60


In [13]:
# Motion filtering: drop epochs with Motion_p95 above (subject-wise) this percentile.
# Set to None to disable. Typical: 0.90 (90th percentile)
MOTION_PCTL = 0.90


In [14]:
# Output dir
OUT_DIR = Path("./out")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [15]:
# Column mapping for your headers
# -----------------------------
CANON_MAP: Dict[str, str] = {
    "zeit": "TIME",
    "bemerkung": "NOTE",
    "u_13_scl": "SCL",     # tonic EDA
    "u_14_scr": "SCR",     # phasic EDA proxy
    "u_15_tem": "TEMP",    # skin temperature
    "u_16_bvp": "BVP",     # PPG waveform
    "u_17_pva": "PVA",     # pulse volume amplitude
    "u_18_puls": "HR",     # bpm
    "u_19_mot": "MOTION",  # motion
}
def canon(col: str) -> str:
    return CANON_MAP.get(str(col).strip().lower(), str(col))


In [16]:
# Helpers: normalization & time parser
# -----------------------------
def _norm(s: str) -> str:
    s = (s or "").strip().lower()
    s = unicodedata.normalize("NFKD", s).encode("ascii","ignore").decode("ascii")
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def make_filekey(name: str) -> str:
    stem = Path(str(name)).stem
    stem = stem.split("(")[0]  # drop "(Session000x)" etc.
    return _norm(stem)

TIME_RE = re.compile(r"^(?P<h>\d{1,2}):(?P<m>\d{2}):(?P<s>\d{2})(?:[.,](?P<f>\d+))?$")
def parse_zeit_to_seconds(x) -> Optional[float]:
    if pd.isna(x): return None
    s = str(x).strip()
    m = TIME_RE.match(s)
    if not m: return None
    h = int(m.group("h")); m_ = int(m.group("m")); s_ = int(m.group("s"))
    frac = m.group("f"); f = 0.0 if frac is None else float("0."+frac.replace(",", ""))
    return float(h*3600 + m_*60 + s_) + f


In [17]:
# Optional: candidate mapping (file→subject_id)
# -----------------------------
FILEKEY_TO_SUBJECT: Dict[str, str] = {}
if CANDIDATE_LIST:
    map_df = pd.read_excel(CANDIDATE_LIST)
    # guess columns
    def _find(colnames, options):
        for c in map_df.columns:
            if str(c).strip().lower() in options: return c
        return None
    col_code = _find(map_df.columns, {"code","subject","subject_id","id","candidate","candidate_code"})
    col_file = _find(map_df.columns, {"file","filename","stem","name","file_name"})
    if col_code is None:
        raise ValueError("Mapping needs a subject code column (code/subject/subject_id/id/...)")
    if col_file is None:
        map_df["__filekey__"] = map_df[col_code].astype(str).map(make_filekey)
    else:
        map_df["__filekey__"] = map_df[col_file].astype(str).map(make_filekey)
    map_df["__subject__"] = map_df[col_code].astype(str).map(_norm)
    FILEKEY_TO_SUBJECT = dict(zip(map_df["__filekey__"], map_df["__subject__"]))



ValueError: Mapping needs a subject code column (code/subject/subject_id/id/...)

In [18]:
# Subject ID extraction
# -----------------------------
def extract_subject_id(path: str) -> str:
    """
    Preferred: use mapping file (if provided).
    Otherwise: filename stem (cleaned) becomes subject_id.
    """
    key = make_filekey(Path(path).stem)
    sid = FILEKEY_TO_SUBJECT.get(key)
    if sid: return sid[:24]
    return key[:24]



In [19]:
# Robust readers for weird .xls
# -----------------------------
def _read_with_engines(path: str) -> pd.DataFrame:
    ext = Path(path).suffix.lower()
    # 1) by extension
    try:
        if ext == ".csv":
            return pd.read_csv(path, encoding="utf-8", engine="python")
        if ext in (".xlsx",".xlsm"):
            return pd.read_excel(path, engine="openpyxl")
        if ext == ".xls":
            return pd.read_excel(path, engine="xlrd")
    except Exception:
        pass
    # 2) try openpyxl (misnamed xlsx)
    try:
        return pd.read_excel(path, engine="openpyxl")
    except Exception:
        pass
    # 3) pandas auto
    try:
        return pd.read_excel(path)
    except Exception:
        pass
    # 4) HTML table (some .xls are HTML)
    try:
        tables = pd.read_html(path, flavor="lxml")
        if tables:
            tables.sort(key=lambda d: d.shape[1], reverse=True)
            return tables[0]
    except Exception:
        pass
    # 5) CSV heuristics
    for enc in ("utf-8","latin-1","cp1252"):
        for sep in (",",";","\t","|"):
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, engine="python")
                if df.shape[1] > 1 or df.shape[0] > 1:
                    return df
            except Exception:
                continue
    raise ValueError(f"Unable to read file with any fallback: {path}")

def read_any(path: str) -> pd.DataFrame:
    df = _read_with_engines(path)
    if df.shape[1] == 1:
        # try to split single column text by common separators
        col = df.columns[0]
        sample = " ".join(df[col].astype(str).head(8).tolist())
        for sep in (";",";", "\t","|"):
            if sep in sample:
                return df[col].str.split(sep, expand=True)
    return df



In [20]:
# Tidy converter
# -----------------------------
def read_and_tidy(path: str, subject_id: str, session_type: str, method: Optional[str]) -> pd.DataFrame:
    df = read_any(path)
    df.columns = [canon(c) for c in df.columns]

    # Time
    if "TIME" in df.columns:
        times = df["TIME"].apply(parse_zeit_to_seconds)
        df = df.loc[times.notna()].copy()
        df["timestamp"] = times.loc[times.notna()].astype(float)
    else:
        # fallback: first numeric column as time, else synthetic 1 Hz
        numeric_cols = [c for c in df.columns if pd.to_numeric(df[c], errors="coerce").notna().sum() > 0]
        if numeric_cols:
            first = numeric_cols[0]
            df = df[pd.to_numeric(df[first], errors="coerce").notna()].copy()
            df["timestamp"] = pd.to_numeric(df[first], errors="coerce").astype(float)
        else:
            df["timestamp"] = np.arange(len(df), dtype=float)

    value_cols = [c for c in df.columns if c not in ("TIME","NOTE","timestamp")]
    meth = None if (method in ("", "none", None)) else method

    rows = []
    ts = df["timestamp"].to_numpy(float)
    for col in value_cols:
        # decimal comma → dot, then numeric
        series = df[col].astype(str).str.replace(",", ".", regex=False)
        xs = pd.to_numeric(series, errors="coerce").to_numpy(float)
        mask = ~np.isnan(xs) & ~np.isnan(ts)
        rows.extend([
            (subject_id, session_type, meth, col, float(t), float(v), path)
            for t, v in zip(ts[mask], xs[mask])
        ])
    return pd.DataFrame(rows, columns=["subject_id","session_type","method","signal","timestamp","value","file_path"])



In [21]:
# Feature computation (epoch-aware)
# -----------------------------
def simple_peaks(x: np.ndarray, height: float | None = None):
    # lightweight peak detector (use scipy.signal.find_peaks if installed)
    try:
        from scipy.signal import find_peaks
        p, _ = find_peaks(x, height=height)
        return p
    except Exception:
        x = np.asarray(x, dtype=float)
        peaks = []
        for i in range(1, len(x)-1):
            if x[i] > x[i-1] and x[i] > x[i+1] and (height is None or x[i] >= height):
                peaks.append(i)
        return np.asarray(peaks, dtype=int)

def compute_features(tidy: pd.DataFrame, epoch_s: float | None = None) -> pd.DataFrame:
    if tidy.empty: return pd.DataFrame()
    out = []
    keys = ["subject_id","session_type","method","signal"]
    tidy = tidy.sort_values(keys + ["timestamp"])
    for (subj, sess, meth, sig), g in tidy.groupby(keys, dropna=False):
        t = g["timestamp"].to_numpy(float)
        x = g["value"].to_numpy(float)
        if epoch_s and epoch_s > 0:
            bins = np.floor((t - t.min())/epoch_s).astype(int)
            g = g.assign(epoch=bins)
            segs = g.groupby("epoch")
        else:
            segs = [(0, g.assign(epoch=0))]
        for ep, sg in segs:
            ts = sg["timestamp"].to_numpy(float)
            xs = sg["value"].to_numpy(float)
            dur = float(ts.max()-ts.min()) if ts.size>1 else 0.0
            row = {"subject_id":subj,"session_type":sess,"method":meth,"signal":sig,
                   "epoch": int(ep) if epoch_s else None, "duration_s": dur}
            u = sig.upper()
            if u == "HR":
                row["HR_mean"] = float(np.nanmean(xs))
                row["HR_std"]  = float(np.nanstd(xs, ddof=1)) if xs.size>1 else float("nan")
            elif u in ("BVP","PPG","ECG"):
                if dur > 0:
                    peaks = simple_peaks(xs, height=np.nanmean(xs))
                    row["HR_est_bpm"] = float((len(peaks)/dur)*60.0) if dur>0 else float("nan")
                    row["peak_count"]  = int(len(peaks))
            elif u == "SCL":
                row["SCL_mean"] = float(np.nanmean(xs))
                if ts.size >= 2:
                    A = np.vstack([ts, np.ones_like(ts)]).T
                    m, c = np.linalg.lstsq(A, xs, rcond=None)[0]
                    row["SCL_slope_per_min"] = float(m*60.0)
            elif u == "SCR":
                pos = np.maximum(xs, 0.0)
                row["SCR_pos_area"] = float(trapezoid(pos, x=ts)) if ts.size>1 else float(np.nansum(pos))
                row["SCR_mean"] = float(np.nanmean(xs))
            elif u == "PVA":
                row["PVA_mean"] = float(np.nanmean(xs))
                row["PVA_std"]  = float(np.nanstd(xs, ddof=1)) if xs.size>1 else float("nan")
            elif u == "TEMP":
                row["Temp_mean"] = float(np.nanmean(xs))
                if ts.size >= 2:
                    A = np.vstack([ts, np.ones_like(ts)]).T
                    m, c = np.linalg.lstsq(A, xs, rcond=None)[0]
                    row["Temp_slope_per_min"] = float(m*60.0)
            elif u == "MOTION":
                row["Motion_mean"] = float(np.nanmean(xs))
                row["Motion_p95"]  = float(np.nanpercentile(xs, 95)) if xs.size>1 else float("nan")
            out.append(row)
    return pd.DataFrame(out)



In [22]:
# Batch processing
# -----------------------------
def list_files(folder: str) -> List[str]:
    files = []
    for pat in ("*.xls","*.xlsx","*.csv"):
        files += glob.glob(os.path.join(folder, pat))
    return sorted(files)

def process_folder(folder: str, session_type: str, method: Optional[str]) -> List[pd.DataFrame]:
    files = list_files(folder)
    print(f"{session_type} / {method}: {len(files)} files")
    out = []
    for fp in files:
        sid = extract_subject_id(fp)
        try:
            tidy = read_and_tidy(fp, sid, session_type, method)
            out.append(tidy)
        except Exception as e:
            print(f"  ! Failed: {fp}: {type(e).__name__}: {e}")
    return out



In [23]:
# RUN
# -----------------------------
all_tidy = []
all_tidy += process_folder(FOLDER_STRESS,       "stress",     None)
all_tidy += process_folder(FOLDER_CONVENTIONAL, "meditation", "conventional")
all_tidy += process_folder(FOLDER_CUSTOM,       "meditation", "custom")

tidy_df = pd.concat(all_tidy, ignore_index=True) if all_tidy else pd.DataFrame(
    columns=["subject_id","session_type","method","signal","timestamp","value","file_path"]
)
print("Total tidy rows:", len(tidy_df))

# Features (epoch-based)
features_df = compute_features(tidy_df, epoch_s=(None if EPOCH_S in (0, None) else float(EPOCH_S)))
print("Total feature rows:", len(features_df))

# Save combined CSVs
tidy_csv = OUT_DIR / "all_sessions_tidy.csv"
feat_csv  = OUT_DIR / ("all_sessions_features_epoch.csv" if EPOCH_S else "all_sessions_features.csv")
tidy_df.to_csv(tidy_csv, index=False)
features_df.to_csv(feat_csv, index=False)
print("Saved:", tidy_csv)
print("Saved:", feat_csv)



stress / None: 0 files
meditation / conventional: 0 files
meditation / custom: 6 files
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
  ! Failed: ./VR meditation\1.xls: ValueError: Unable to read file with any fallback: ./VR meditation\1.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
  ! Failed: ./VR meditation\2.xls: ValueError: Unable to read file with any fallback: ./VR meditation\2.xls
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Total tidy rows: 372057
Total feature rows: 168
Saved: out\all_sessions_tidy.csv
Saved: out\all_sessions_features_epoch.csv


In [24]:
# Optional motion filtering
# -----------------------------
feat = features_df.copy()
if MOTION_PCTL is not None and "Motion_p95" in feat.columns:
    thr = feat.groupby("subject_id")["Motion_p95"].quantile(MOTION_PCTL)
    feat = feat.merge(thr.rename("motion_thr"), on="subject_id", how="left")
    feat = feat[(feat["Motion_p95"].isna()) | (feat["Motion_p95"] <= feat["motion_thr"])].drop(columns=["motion_thr"])



In [25]:
def epoch_curve(df, signal, value_col, title):
    d = df[(df["signal"].str.upper()==signal) & (df["session_type"]=="meditation")]
    if d.empty or d["epoch"].isna().all():
        print("No epoch data"); return
    # pick delta if present
    val = value_col if value_col in d.columns else d.columns[d.columns.str.endswith("_delta")][0]
    g = (d.groupby(["method","epoch"])[val]
           .agg(["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
           .rename(columns={"<lambda_0>":"q25","<lambda_1>":"q75"})
           .reset_index())
    fig, ax = plt.subplots(figsize=(7,4))
    for m, gm in g.groupby("method"):
        ax.plot(gm["epoch"], gm["median"], label=m)
        ax.fill_between(gm["epoch"], gm["q25"], gm["q75"], alpha=0.2)
    ax.set_title(title); ax.set_xlabel("Epoch (min if epoch=60s)"); ax.legend()
    plt.tight_layout(); plt.show()

epoch_curve(scl, "SCL", "SCL_mean_delta", "ΔSCL over time during meditation")
epoch_curve(hr,  "HR",  "HR_mean_delta",  "ΔHR over time during meditation")


NameError: name 'scl' is not defined

In [ ]:
# Baseline center (first epoch)
# -----------------------------
def baseline_center_epoch(df, signal, metric_cols, base_epochs=1):
    d = df[(df["signal"].str.upper()==signal) & (df["session_type"]=="meditation")].copy()
    d = d.dropna(subset=["epoch"])
    if d.empty: return d
    keys = ["subject_id","session_type","method"]
    d = d.sort_values(keys + ["epoch"]).copy()
    min_ep = d.groupby(keys)["epoch"].transform("min")
    d["is_baseline"] = d["epoch"] <= (min_ep + (base_epochs - 1))
    base = d.loc[d["is_baseline"]].groupby(keys)[metric_cols].mean().reset_index()
    base = base.rename(columns={c: f"{c}_base" for c in metric_cols})
    d = d.merge(base, on=keys, how="left")
    for c in metric_cols:
        d[f"{c}_delta"] = d[c] - d[f"{c}_base"]
    return d

scl = baseline_center_epoch(feat, "SCL", ["SCL_mean"], base_epochs=1)
pva = baseline_center_epoch(feat, "PVA", ["PVA_mean"], base_epochs=1)
hr  = baseline_center_epoch(feat, "HR",  ["HR_mean"],  base_epochs=1)
tmp = baseline_center_epoch(feat, "TEMP",["Temp_slope_per_min"], base_epochs=1)  # slope doesn't need delta, but kept consistent



In [ ]:
# Pairing sanity check
# -----------------------------
med = feat.query("session_type=='meditation'").copy()
pair_table = med.groupby(["subject_id","method"]).size().unstack(fill_value=0)
print("\nMeditation files per subject/method:\n", pair_table)
paired_subjs = (med.dropna(subset=["method"])
                  .groupby("subject_id")["method"].nunique())
paired_ids = paired_subjs[paired_subjs>=2].index.tolist()
print("Subjects with BOTH methods:", paired_ids)


# -----------------------------
# Build per-subject tables (means across epochs)
# -----------------------------
def subject_means(df, signal, cols):
    if df is None or df.empty: 
        return pd.DataFrame(columns=["subject_id","method"]+cols)
    d = df[df["signal"].str.upper()==signal]
    return (d.groupby(["subject_id","method"], dropna=False)[cols]
              .mean().reset_index())

scl_col = "SCL_mean_delta" if "SCL_mean_delta" in scl.columns else "SCL_mean"
pva_col = "PVA_mean_delta" if "PVA_mean_delta" in pva.columns else "PVA_mean"
hr_col  = "HR_mean_delta"  if "HR_mean_delta"  in hr.columns  else "HR_mean"
tmp_col = "Temp_slope_per_min"  # slope itself is the metric

scl_tbl = subject_means(scl, "SCL",  [scl_col])
pva_tbl = subject_means(pva, "PVA",  [pva_col])
hr_tbl  = subject_means(hr,  "HR",   [hr_col])
tmp_tbl = subject_means(tmp, "TEMP", [tmp_col])

# Save per-signal comparison CSVs (meditation only)
scl_tbl.to_csv(OUT_DIR / "compare_scl.csv", index=False)
pva_tbl.to_csv(OUT_DIR / "compare_pva.csv", index=False)
hr_tbl.to_csv(OUT_DIR / "compare_hr.csv", index=False)
tmp_tbl.to_csv(OUT_DIR / "compare_temp.csv", index=False)



Meditation files per subject/method:
 method      custom
subject_id        
10              41
4               48
6               35
9               41
Subjects with BOTH methods: []


In [ ]:
# Paired within-subjects (if possible)
# -----------------------------
def paired_compare(tbl, col, a="custom", b="conventional"):
    wide = tbl.pivot(index="subject_id", columns="method", values=col).dropna()
    need = [a,b]
    if not set(need).issubset(wide.columns):
        return {"n":0,"mean_diff":np.nan,"t":np.nan,"p":np.nan,"cohens_dz":np.nan,"ci95_low":np.nan,"ci95_high":np.nan}
    wide = wide.dropna(subset=need)
    x = wide[a]; y = wide[b]
    if len(wide) < 2:
        return {"n":len(wide), "mean_diff":(x-y).mean() if len(wide)==1 else np.nan,
                "t":np.nan,"p":np.nan,"cohens_dz":np.nan,"ci95_low":np.nan,"ci95_high":np.nan}
    t, p = stats.ttest_rel(x, y, nan_policy="omit")
    diff = (x - y)
    dz = diff.mean() / (diff.std(ddof=1) + 1e-12)
    ci = stats.t.interval(0.95, len(diff)-1, loc=diff.mean(), scale=stats.sem(diff))
    return {"n":len(diff), "mean_diff":diff.mean(), "t":t, "p":p, "cohens_dz":dz, "ci95_low":ci[0], "ci95_high":ci[1]}

paired_scl = paired_compare(scl_tbl.rename(columns={scl_col:"metric"}), "metric")
paired_pva = paired_compare(pva_tbl.rename(columns={pva_col:"metric"}), "metric")
paired_hr  = paired_compare(hr_tbl.rename(columns={hr_col:"metric"}),   "metric")
paired_tmp = paired_compare(tmp_tbl.rename(columns={tmp_col:"metric"}), "metric")

print("\nPAIRED (custom - conventional):")
print("ΔSCL:", paired_scl)
print("ΔPVA:", paired_pva)
print("ΔHR :", paired_hr)
print("Temp slope:", paired_tmp)




PAIRED (custom - conventional):
ΔSCL: {'n': 0, 'mean_diff': nan, 't': nan, 'p': nan, 'cohens_dz': nan, 'ci95_low': nan, 'ci95_high': nan}
ΔPVA: {'n': 0, 'mean_diff': nan, 't': nan, 'p': nan, 'cohens_dz': nan, 'ci95_low': nan, 'ci95_high': nan}
ΔHR : {'n': 0, 'mean_diff': nan, 't': nan, 'p': nan, 'cohens_dz': nan, 'ci95_low': nan, 'ci95_high': nan}
Temp slope: {'n': 0, 'mean_diff': nan, 't': nan, 'p': nan, 'cohens_dz': nan, 'ci95_low': nan, 'ci95_high': nan}


In [26]:
# Between-subjects (Welch) — if no pairs
# -----------------------------
def welch_g(tbl, col):
    a = tbl.loc[tbl["method"]=="custom", col].dropna().to_numpy()
    b = tbl.loc[tbl["method"]=="conventional", col].dropna().to_numpy()
    if len(a)<2 or len(b)<2:
        return {"n_custom":len(a),"n_conv":len(b),"t":np.nan,"p":np.nan,"g":np.nan,"ci_low":np.nan,"ci_high":np.nan}
    t, p = stats.ttest_ind(a, b, equal_var=False)
    na, nb = len(a), len(b)
    sa2, sb2 = a.var(ddof=1), b.var(ddof=1)
    sp = np.sqrt(((na-1)*sa2 + (nb-1)*sb2) / (na+nb-2))
    d = (a.mean() - b.mean()) / (sp + 1e-12)
    J = 1 - 3/(4*(na+nb)-9)  # Hedges correction
    g = J*d
    se = np.sqrt(sa2/na + sb2/nb)
    df = (sa2/na + sb2/nb)**2 / ((sa2**2)/((na**2)*(na-1)) + (sb2**2)/((nb**2)*(nb-1)))
    ci = stats.t.interval(0.95, df, loc=(a.mean()-b.mean()), scale=se)
    return {"n_custom":na,"n_conv":nb,"t":t,"p":p,"g":g,"ci_low":ci[0],"ci_high":ci[1]}

if paired_scl["n"] < 2 or paired_pva["n"] < 2 or paired_hr["n"] < 2:
    print("\nBETWEEN-SUBJECTS (Welch) — used if pairs are insufficient:")
    print("SCL:",  welch_g(scl_tbl, scl_col))
    print("PVA:",  welch_g(pva_tbl, pva_col))
    print("HR :",  welch_g(hr_tbl,  hr_col))
    print("TMP:",  welch_g(tmp_tbl, tmp_col))



NameError: name 'paired_scl' is not defined

In [27]:
# Plots (paired spaghetti if possible; otherwise group boxplots)
# -----------------------------
def paired_plot(tbl, col, title):
    wide = tbl.pivot(index="subject_id", columns="method", values=col).dropna()
    if wide.empty:
        print(f"No pairs for {title}"); return
    fig, ax = plt.subplots(figsize=(5,5))
    for _, row in wide.iterrows():
        ax.plot([0,1], [row.get("conventional"), row.get("custom")], marker="o", alpha=0.6)
    ax.set_xticks([0,1], ["Conventional","Custom"])
    ax.set_title(title); ax.axhline(0, ls="--", lw=1, alpha=0.5)
    plt.tight_layout(); plt.show()

def group_box(tbl, col, title, ylabel):
    df = tbl[["method", col]].dropna()
    if df.empty:
        print(f"No data for {title}"); return
    fig, ax = plt.subplots(figsize=(5,5))
    order = ["conventional","custom"] if set(df["method"]) >= {"conventional","custom"} else sorted(df["method"].unique())
    data = [df[df["method"]==m][col] for m in order]
    ax.boxplot(data, labels=[m.capitalize() for m in order], showfliers=False)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.axhline(0, ls="--", lw=1, alpha=0.5)
    plt.tight_layout(); plt.show()

# Try paired first; if no pairs, show group boxes
if paired_scl["n"] >= 2:
    paired_plot(scl_tbl.rename(columns={scl_col:"metric"}), "metric", f"{scl_col}: Custom vs Conventional")
else:
    group_box(scl_tbl, scl_col, f"{scl_col}: Group Comparison", "µS or ΔµS")

if paired_pva["n"] >= 2:
    paired_plot(pva_tbl.rename(columns={pva_col:"metric"}), "metric", f"{pva_col}: Custom vs Conventional")
else:
    group_box(pva_tbl, pva_col, f"{pva_col}: Group Comparison", "a.u. or Δ")

if paired_hr["n"] >= 2:
    paired_plot(hr_tbl.rename(columns={hr_col:"metric"}), "metric", f"{hr_col}: Custom vs Conventional")
else:
    group_box(hr_tbl, hr_col, f"{hr_col}: Group Comparison", "bpm or Δbpm")

# Optional time-course (median ± IQR) for ΔSCL and ΔHR during meditation
def epoch_curve(df, signal, value_col, title):
    d = df[(df["signal"].str.upper()==signal) & (df["session_type"]=="meditation")]
    if d.empty or d["epoch"].isna().all() or value_col not in d.columns:
        print("No epoch data for", title); return
    g = (d.groupby(["method","epoch"])[value_col]
           .agg(["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
           .rename(columns={"<lambda_0>":"q25","<lambda_1>":"q75"})
           .reset_index())
    fig, ax = plt.subplots(figsize=(7,4))
    for m, gm in g.groupby("method"):
        ax.plot(gm["epoch"], gm["median"], label=m)
        ax.fill_between(gm["epoch"], gm["q25"], gm["q75"], alpha=0.2)
    ax.set_title(title); ax.set_xlabel("Epoch (min if epoch=60s)"); ax.legend()
    plt.tight_layout(); plt.show()

epoch_curve(scl, "SCL", "SCL_mean_delta" if "SCL_mean_delta" in scl.columns else "SCL_mean",
            "SCL over time during meditation")
epoch_curve(hr,  "HR",  "HR_mean_delta"  if "HR_mean_delta"  in hr.columns  else "HR_mean",
            "HR over time during meditation")

print("\nDone. CSVs are in:", OUT_DIR.resolve())

NameError: name 'paired_scl' is not defined

In [28]:
metric_map = {
    "SCL":  ["SCL_mean","SCL_slope_per_min","SCL_mean_delta"],
    "PVA":  ["PVA_mean","PVA_std","PVA_mean_delta"],
    "HR":   ["HR_mean","HR_std","HR_mean_delta"],
}
per_signal = {}
for sig, cols in metric_map.items():
    d = feat_med[feat_med["signal"].str.upper()==sig]
    keep = ["subject_id","method"] + [c for c in cols if c in d.columns]
    if len(keep) > 2:
        per_signal[sig] = d[keep].groupby(["subject_id","method"]).mean().reset_index()
        display(sig, per_signal[sig].head())


NameError: name 'feat_med' is not defined